# 05 · Selección del modelo ganador, métrica de negocio y Model Registry

1. Ganador vs segundo mejor con la CV (media, desviación y diferencia pareada por pliegue).
2. Evaluación **una sola vez** de ambos en el hold-out (2026-03 → 2026-08).
3. Métrica de negocio con un umbral elegido por costo **sólo con predicciones fuera de pliegue del train**.
4. Costo computacional y explicabilidad.
5. Registro en MLflow: `ecu911-modelo@champion` y `@challenger`; export de corridas a `docs/corridas.csv`.

In [ ]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore")
import src  # noqa: F401  (en Windows carga LightGBM antes que pyarrow)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src.datos import *
FIG = RAIZ / "docs" / "figuras"; FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

In [ ]:
import time, mlflow
from mlflow import MlflowClient
from sklearn.base import clone
from sklearn.metrics import precision_recall_curve
from src.pipeline import (ReglaEstacional, pr_auc, costo, umbral_costo_minimo, metricas_negocio,
                          COSTO_FN, COSTO_FP, construir, candidatos)
from src.experimento import configurar, commit_git, EXPERIMENTO, MODELO_REGISTRO
from mlflow.models import infer_signature
configurar(); client = MlflowClient()
obt = cargar_obt(); MD = meta(obt)
datos = modelable(construir_dataset(obt))
train, test = dividir(datos)
X, y = train[FEATURES], train["y"]
Xt, yt = test[FEATURES], test["y"]
cv = CVTemporal(train.fecha)
print(MD, f"| train={len(X):,} test={len(Xt):,} (positivos test {yt.mean():.2%})")

## 1. Ranking por CV (todas las corridas con pipeline registrado)

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENTO])
elegibles = runs[(runs["tags.fase"].isin(["candidato", "tuning"])) & (runs["tags.rol"].fillna("padre") == "padre")
                 & runs["tags.familia"].isin(["lineal", "bagging", "boosting", "red_neuronal"])].copy()
elegibles["version"] = np.where(elegibles["tags.fase"] == "tuning", "ajustado", "default")
elegibles = elegibles.sort_values("metrics.cv_pr_auc_media", ascending=False)
cols = ["tags.modelo", "version", "metrics.cv_pr_auc_media", "metrics.cv_pr_auc_desv", "metrics.train_pr_auc",
        "metrics.brecha_train_cv", "metrics.segundos_entrenamiento", "run_id"]
display(elegibles[cols].head(10))
mejores = elegibles.drop_duplicates("tags.modelo")          # mejor versión de cada modelo
ganador, segundo = mejores.iloc[0], mejores.iloc[1]
ref = runs[runs["tags.modelo"].isin(["regla_estacional", "dummy_prior", "chronos_bolt_small"])][
    ["tags.modelo", "metrics.cv_pr_auc_media", "metrics.cv_pr_auc_desv"]]
display(ref)
print(f"Ganador: {ganador['tags.modelo']} ({ganador.version}) run={ganador.run_id}")
print(f"Segundo: {segundo['tags.modelo']} ({segundo.version}) run={segundo.run_id}")

In [ ]:
fg = np.array([ganador[f"metrics.cv_pr_auc_fold{i}"] for i in range(5)])
fs = np.array([segundo[f"metrics.cv_pr_auc_fold{i}"] for i in range(5)])
dif = fg - fs
comp_cv = pd.DataFrame({"ganador": fg, "segundo": fs, "diferencia": dif}, index=[f"pliegue {i}" for i in range(5)])
display(comp_cv.round(4))
dmed, dstd = fg.mean() - fs.mean(), max(fg.std(), fs.std())
print(f"Diferencia de medias = {dmed:+.4f} | desviación entre pliegues = {dstd:.4f} → "
      f"{'SUPERA' if dmed > dstd else 'NO supera'} la variación entre pliegues")
print(f"Diferencia pareada: media {dif.mean():+.4f}, desv {dif.std():.4f}, gana en {(dif > 0).sum()}/5 pliegues")

## 2. Umbral de decisión (sólo train) y evaluación única en el hold-out

El costo de negocio supone **1 pico no anticipado (FN) = 5 falsas alertas (FP)**: una falsa alerta cuesta una
revisión humana; un pico no anticipado obliga a coordinar de forma reactiva (PSet 1). El umbral que minimiza
`5·FN + 1·FP` se elige con predicciones **fuera de pliegue** (OOF) de la CV del train y luego se aplica tal cual al test.

In [ ]:
def oof(modelo_pipe):
    s = pd.Series(np.nan, index=X.index)
    for a, b in cv.split():
        s.iloc[b] = clone(modelo_pipe).fit(X.iloc[a], y.iloc[a]).predict_proba(X.iloc[b])[:, 1]
    return s.dropna()

modelos, umbrales, oofs = {}, {}, {}
for rol, r in [("ganador", ganador), ("segundo", segundo)]:
    modelos[rol] = mlflow.sklearn.load_model(f"runs:/{r.run_id}/modelo")
    oofs[rol] = oof(modelos[rol])
    umbrales[rol] = umbral_costo_minimo(y.loc[oofs[rol].index].values, oofs[rol].values)
    c = costo(y.loc[oofs[rol].index], (oofs[rol] >= umbrales[rol]).astype(int))
    print(f"{rol}: umbral OOF = {umbrales[rol]:.4f} | OOF recall={c['recall']:.3f} precision={c['precision']:.3f} "
          f"costo/1000={c['costo_por_1000']:.1f}")
regla_oof = costo(y.loc[oofs['ganador'].index], ReglaEstacional().predict(X.loc[oofs['ganador'].index]))
print(f"regla (OOF mismas filas): recall={regla_oof['recall']:.3f} precision={regla_oof['precision']:.3f} costo/1000={regla_oof['costo_por_1000']:.1f}")

In [ ]:
# ===== Evaluación ÚNICA en test =====
y_regla_t = ReglaEstacional().predict(Xt)
score_t, t_inf = {}, {}
for rol in modelos:
    t0 = time.perf_counter(); score_t[rol] = modelos[rol].predict_proba(Xt)[:, 1]; t_inf[rol] = time.perf_counter() - t0
score_t["regla_estacional"] = ReglaEstacional().predict_proba(Xt)[:, 1]
filas = []
for rol in ["ganador", "segundo"]:
    nb_ = metricas_negocio(yt.values, score_t[rol], umbrales[rol], y_regla_t)
    filas.append({"rol": rol, "modelo": (ganador if rol == "ganador" else segundo)["tags.modelo"],
                  "test_pr_auc": pr_auc(yt, score_t[rol]), "umbral": umbrales[rol], **nb_,
                  "ms_inferencia_por_1000": 1000 * t_inf[rol] / len(Xt) * 1000})
filas.append({"rol": "referencia", "modelo": "regla_estacional", "test_pr_auc": pr_auc(yt, score_t["regla_estacional"]),
              **{k: v for k, v in costo(yt, y_regla_t).items() if k in ("recall", "precision")},
              "costo_regla_por_1000": costo(yt, y_regla_t)["costo_por_1000"], "alertas_regla": int(y_regla_t.sum())})
filas.append({"rol": "referencia", "modelo": "prevalencia (dummy)", "test_pr_auc": yt.mean()})
evaluacion = pd.DataFrame(filas)
evaluacion.to_csv(RAIZ / "docs" / "evaluacion_test.csv", index=False)
evaluacion.T

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for rol, col in [("ganador", "#4C72B0"), ("segundo", "#DD8452"), ("regla_estacional", "#55A868")]:
    p, r, _ = precision_recall_curve(yt, score_t[rol]); ax[0].plot(r, p, label=f"{rol} (AP={pr_auc(yt, score_t[rol]):.3f})", c=col)
cr = costo(yt, y_regla_t); ax[0].scatter(cr["recall"], cr["precision"], c="#55A868", marker="X", s=90, zorder=5, label="regla (su umbral)")
cg = costo(yt, (score_t["ganador"] >= umbrales["ganador"]).astype(int))
ax[0].scatter(cg["recall"], cg["precision"], c="#4C72B0", marker="*", s=160, zorder=5, label="ganador (umbral de costo)")
ax[0].axhline(yt.mean(), c="grey", ls=":", label="prevalencia"); ax[0].set(title="Curva precisión–recall en test", xlabel="recall", ylabel="precisión"); ax[0].legend(fontsize=8)
us = np.linspace(0.02, 0.6, 60)
ax[1].plot(us, [costo(yt, (score_t["ganador"] >= u).astype(int))["costo_por_1000"] for u in us], label="ganador")
ax[1].axhline(cr["costo_por_1000"], c="#55A868", ls="--", label="regla estacional")
ax[1].axhline(1000 * COSTO_FN * yt.mean(), c="grey", ls=":", label="no alertar nunca")
ax[1].axvline(umbrales["ganador"], c="k", ls="--", lw=.8, label="umbral elegido (OOF)")
ax[1].set(title=f"Costo por 1000 cantón-días en test (FN={COSTO_FN:.0f}, FP={COSTO_FP:.0f})", xlabel="umbral de probabilidad", ylabel="costo/1000"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "seleccion_test.png", bbox_inches="tight"); plt.show()

### Desempeño por estrato de volumen del cantón (PSet 1: riesgo en cantones pequeños)

In [ ]:
est = pd.qcut(test.media_prev_28d, 5, labels=["Q1 (bajo)", "Q2", "Q3", "Q4", "Q5 (alto)"])
alerta_g = (score_t["ganador"] >= umbrales["ganador"]).astype(int)
por_estrato = []
for e in est.cat.categories:
    m = (est == e).values
    cg, crr = costo(yt[m], alerta_g[m]), costo(yt[m], y_regla_t[m])
    por_estrato.append({"estrato": e, "filas": int(m.sum()), "prevalencia": yt[m].mean(),
                        "pr_auc_ganador": pr_auc(yt[m], score_t["ganador"][m]), "pr_auc_regla": pr_auc(yt[m], score_t["regla_estacional"][m]),
                        "recall_ganador": cg["recall"], "precision_ganador": cg["precision"], "recall_regla": crr["recall"]})
por_estrato = pd.DataFrame(por_estrato); por_estrato

## 3. Costo y explicabilidad

In [ ]:
from sklearn.inspection import permutation_importance
costo_tabla = pd.DataFrame({
    "modelo": [ganador["tags.modelo"], segundo["tags.modelo"]],
    "seg_entrenamiento_train_completo": [ganador["metrics.segundos_entrenamiento"], segundo["metrics.segundos_entrenamiento"]],
    "ms_inferencia_por_1000_filas": [1000 * t_inf["ganador"] / len(Xt) * 1000, 1000 * t_inf["segundo"] / len(Xt) * 1000]})
display(costo_tabla)
print(f"Un día de producción = 224 cantones → inferencia < {costo_tabla.ms_inferencia_por_1000_filas.max()*0.224:.1f} ms")
mi = Xt.sample(20000, random_state=SEMILLA)
pi = permutation_importance(modelos["ganador"], mi, yt.loc[mi.index], scoring="average_precision", n_repeats=5,
                            random_state=SEMILLA, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values()
imp.tail(12).plot.barh(figsize=(7, 4.5), xerr=pd.Series(pi.importances_std, index=FEATURES)[imp.tail(12).index])
plt.title("Importancia por permutación (caída de PR-AUC, test)"); plt.tight_layout(); plt.savefig(FIG / "seleccion_importancia.png", bbox_inches="tight"); plt.show()

In [ ]:
# Explicación local: contribuciones (SHAP de árboles) de una alerta concreta, si el modelo las expone
est_g = modelos["ganador"].named_steps["modelo"]; prep = modelos["ganador"].named_steps["prep"]
i = int(np.argmax(score_t["ganador"] * yt.values))      # la alerta correcta con mayor probabilidad
fila = test.iloc[[i]]
Z = prep.transform(fila[FEATURES]); nombres = prep.get_feature_names_out()
contrib = None
if hasattr(est_g, "booster_"):
    contrib = est_g.booster_.predict(Z, pred_contrib=True)[0]
elif hasattr(est_g, "get_booster"):
    import xgboost as xgb
    contrib = est_g.get_booster().predict(xgb.DMatrix(Z, feature_names=list(nombres)), pred_contribs=True)[0]
print(f"Cantón {fila.canton.iloc[0]}, t={fila.fecha.iloc[0].date()} → d={fila.fecha_objetivo.iloc[0].date()}, "
      f"P(pico)={score_t['ganador'][i]:.3f}, umbral P90={fila.umbral_objetivo.iloc[0]:.1f}, real={fila.n_objetivo.iloc[0]:.0f}")
if contrib is not None:
    c = pd.Series(contrib[:-1], index=nombres)
    display(c.reindex(c.abs().sort_values(ascending=False).index).head(8).to_frame("contribución (log-odds)"))
    display(fila[["n_emergencias", "media_prev_28d", "tasa_picos_28d", "ratio_regla", "ratio_hoy_28", "es_feriado_obj", "dia_semana_obj"]])

## 4. Corridas finales y Model Registry

In [ ]:
ejemplo = Xt.head(5)
reg = {}
for rol, r, alias in [("ganador", ganador, "champion"), ("segundo", segundo, "challenger")]:
    fila = evaluacion[evaluacion.rol == rol].iloc[0]
    with mlflow.start_run(run_name=f"final_{r['tags.modelo']}_{alias}") as run:
        mlflow.log_params({"semilla": SEMILLA, "umbral_decision": umbrales[rol], "costo_fn": COSTO_FN, "costo_fp": COSTO_FP,
                           "test_desde": str(test.fecha.min().date()), "test_hasta": str(test.fecha.max().date()),
                           "n_test": len(Xt), "run_origen": r.run_id, **MD.como_params()})
        mlflow.log_metrics({"test_pr_auc": float(fila.test_pr_auc), "cv_pr_auc_media": r["metrics.cv_pr_auc_media"],
                            "cv_pr_auc_desv": r["metrics.cv_pr_auc_desv"], "train_pr_auc": r["metrics.train_pr_auc"],
                            "segundos_entrenamiento": r["metrics.segundos_entrenamiento"],
                            **{k: float(fila[k]) for k in ["reduccion_fn_pct", "fn_regla", "fn_modelo_mismo_presupuesto",
                                                            "costo_regla_por_1000", "costo_modelo_por_1000", "recall_umbral",
                                                            "precision_umbral", "alertas_umbral", "alertas_regla",
                                                            "ms_inferencia_por_1000"]}})
        mlflow.set_tags({"fase": "final", "familia": r["tags.familia"], "modelo": r["tags.modelo"], "alias": alias,
                         "git_commit": commit_git()})
        firma = infer_signature(ejemplo, modelos[rol].predict_proba(ejemplo))
        info = mlflow.sklearn.log_model(modelos[rol], artifact_path="modelo", signature=firma, input_example=ejemplo,
                                        pyfunc_predict_fn="predict_proba", registered_model_name=MODELO_REGISTRO)
        version = info.registered_model_version
    client.set_registered_model_alias(MODELO_REGISTRO, alias, version)
    client.set_model_version_tag(MODELO_REGISTRO, version, "umbral_decision", f"{umbrales[rol]:.6f}")
    client.set_model_version_tag(MODELO_REGISTRO, version, "test_pr_auc", f"{fila.test_pr_auc:.6f}")
    reg[alias] = {"modelo": r["tags.modelo"], "run_id_final": run.info.run_id, "version": version,
                  "test_pr_auc": fila.test_pr_auc, "umbral": umbrales[rol]}
client.update_registered_model(MODELO_REGISTRO, description=(
    "ECU 911: probabilidad de que un cantón supere su P90 histórico (mismo día de semana, 52 semanas) en t+3. "
    "Entrada: FEATURES de src/datos.py; salida: P(pico). Alerta si P >= tag umbral_decision."))
registro = pd.DataFrame(reg).T; registro

## 5. Degradación con el horizonte t+7 (PSet 1)

Misma configuración del ganador, mismo esquema (CV temporal con `gap` = 7, mismo hold-out), target a 7 días.

In [ ]:
from sklearn.model_selection import cross_val_score
d7 = modelable(construir_dataset(obt, h=7))
tr7, te7 = dividir(d7, h=7)
cv7 = CVTemporal(tr7.fecha, gap=7)
pipe7 = clone(modelos["ganador"])
s7 = cross_val_score(pipe7, tr7[FEATURES], tr7.y, cv=cv7, scoring="average_precision")
m7 = clone(modelos["ganador"]).fit(tr7[FEATURES], tr7.y)
h7 = {"cv_pr_auc_media": s7.mean(), "cv_pr_auc_desv": s7.std(),
      "test_pr_auc": pr_auc(te7.y, m7.predict_proba(te7[FEATURES])[:, 1]),
      "test_pr_auc_regla": pr_auc(te7.y, te7.ratio_regla), "prevalencia_test": te7.y.mean()}
with mlflow.start_run(run_name=f"final_{ganador['tags.modelo']}_h7"):
    mlflow.log_params({"horizonte": 7, "semilla": SEMILLA, "run_origen": ganador.run_id, **MD.como_params()})
    mlflow.log_metrics(h7)
    mlflow.set_tags({"fase": "final", "familia": ganador["tags.familia"], "modelo": ganador["tags.modelo"],
                     "horizonte": "t+7", "git_commit": commit_git()})
pd.DataFrame({"t+3": {"cv_pr_auc_media": ganador["metrics.cv_pr_auc_media"], "test_pr_auc": reg["champion"]["test_pr_auc"],
                      "test_pr_auc_regla": evaluacion.query("modelo == 'regla_estacional'").test_pr_auc.iloc[0], "prevalencia_test": yt.mean()},
              "t+7": h7}).T

## 6. Export de todas las corridas → `docs/corridas.csv`

In [ ]:
todas = mlflow.search_runs(experiment_names=[EXPERIMENTO], max_results=5000)
todas = todas.drop(columns=[c for c in todas.columns if c.startswith("tags.mlflow.log-model")])
todas.to_csv(RAIZ / "docs" / "corridas.csv", index=False)
print(f"{len(todas)} corridas exportadas")
todas.groupby("tags.fase").size().rename("corridas").to_frame()